In [1]:
import os
import random
import gc

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

from sklearn.linear_model import LinearRegression

SEED = 1337


def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(SEED)



In [2]:
df_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
df_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
df_submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

print(
    f'Training Set Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(f'Test Set Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}')
print(f"Sample Submission Shape = {df_submission.shape}")
print(df_submission.head())




Training Set Shape = (1394, 7) - Patients = 158
Test Set Shape = (18, 7) - Patients = 18
Sample Submission Shape = (1908, 3)
                   Patient_Week   FVC  Confidence
0  ID00126637202218610655908_-3  2000         100
1  ID00126637202218610655908_-2  2000         100
2  ID00126637202218610655908_-1  2000         100
3   ID00126637202218610655908_0  2000         100
4   ID00126637202218610655908_1  2000         100


In [3]:
class Preprocessor:
    def __init__(
        self, df_train, df_test, df_submission, n_folds, shuffle, resize_shape
    ):
        self.df_train = df_train.copy(deep=True)
        self.df_train.sort_values(by=["Patient", "Weeks"], inplace=True)
        self.df_test = df_test.copy(deep=True)
        self.df_submission = df_submission.copy(deep=True)

        self.n_folds = n_folds
        self.shuffle = shuffle
        self.resize_shape = resize_shape

    def _drop_duplicates(self):
        self.df_train["FVC"] = self.df_train.groupby(["Patient", "Weeks"])[
            "FVC"
        ].transform("mean")
        self.df_train["Percent"] = self.df_train.groupby(["Patient", "Weeks"])[
            "Percent"
        ].transform("mean")
        self.df_train.drop_duplicates(inplace=True)
        self.df_train.reset_index(drop=True, inplace=True)

    def _label_encode(self):
        sex_map = {"Male": 0, "Female": 1}
        smoke_map = {"Never smoked": 0, "Ex-smoker": 1, "Currently smokes": 2}

        for df in [self.df_train, self.df_test]:
            df["Sex"] = df["Sex"].map(sex_map)
            if df["Sex"].isna().any():
                df["Sex"] = df["Sex"].fillna(df["Sex"].mode().iloc[0])
            df["Sex"] = df["Sex"].astype("int64")

            df["SmokingStatus"] = df["SmokingStatus"].map(smoke_map)
            if df["SmokingStatus"].isna().any():
                df["SmokingStatus"] = df["SmokingStatus"].fillna(
                    df["SmokingStatus"].mode().iloc[0]
                )
            df["SmokingStatus"] = df["SmokingStatus"].astype("int64")

    def _create_folds(self):
        self.df_train["Sex_SmokingStatus"] = (
            self.df_train["Sex"].astype(str)
            + "_"
            + self.df_train["SmokingStatus"].astype(str)
        )

        for group in self.df_train["Sex_SmokingStatus"].unique():
            patients = self.df_train[self.df_train["Sex_SmokingStatus"] == group][
                "Patient"
            ].unique()

            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)

            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.df_train.loc[
                    self.df_train["Patient"].isin(patient_group), "CV1_Fold"
                ] = fold

        for patient_name in self.df_train["Patient"].unique():
            last2 = self.df_train.loc[
                self.df_train["Patient"] == patient_name, "FVC"
            ].values[-2:]
            mu = last2.mean()
            sd = last2.std()
            if sd == 0:
                z = np.zeros_like(last2, dtype=np.float64)
            else:
                z = (last2 - mu) / sd

            Xw = (
                self.df_train.loc[self.df_train["Patient"] == patient_name, "Weeks"]
                .values[-2:]
                .reshape(-1, 1)
            )

            reg = LinearRegression().fit(Xw, z)
            self.df_train.loc[self.df_train["Patient"] == patient_name, "Intercept"] = (
                reg.intercept_
            )
            self.df_train.loc[self.df_train["Patient"] == patient_name, "Coef"] = (
                reg.coef_[0]
            )

        self.df_train.loc[self.df_train["Coef"] > 0.4, "Cluster"] = 1
        self.df_train.loc[
            (self.df_train["Coef"] <= 0.4) & (self.df_train["Coef"] >= -0.4), "Cluster"
        ] = 2
        self.df_train.loc[self.df_train["Coef"] < -0.4, "Cluster"] = 3

        for group in self.df_train["Cluster"].unique():
            patients = self.df_train[self.df_train["Cluster"] == group][
                "Patient"
            ].unique()

            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)

            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.df_train.loc[
                    self.df_train["Patient"].isin(patient_group), "CV2_Fold"
                ] = fold

        patients = self.df_train["Patient"].unique()
        np.random.seed(SEED)
        np.random.shuffle(patients)
        for fold, patient_group in enumerate(np.array_split(patients, self.n_folds), 1):
            self.df_train.loc[
                self.df_train["Patient"].isin(patient_group), "CV3_Fold"
            ] = fold

        self.df_train.drop(
            columns=["Sex_SmokingStatus", "Intercept", "Coef", "Cluster"], inplace=True
        )

        for c in ["CV1_Fold", "CV2_Fold", "CV3_Fold"]:
            self.df_train[c] = self.df_train[c].astype(np.uint8)

    def _create_baseline_features(self):
        self.df_submission["Type"] = "Test"
        self.df_submission["Patient"] = (
            self.df_submission["Patient_Week"]
            .apply(lambda x: x.split("_")[0])
            .astype(str)
        )
        self.df_submission["Weeks"] = (
            self.df_submission["Patient_Week"]
            .apply(lambda x: x.split("_")[1])
            .astype(int)
        )
        self.df_submission.drop(
            columns=["Patient_Week", "FVC", "Confidence"], inplace=True
        )

        self.df_train["Type"] = "Train"
        self.df_train["Weeks_Passed"] = self.df_train["Weeks"] - self.df_train.groupby(
            "Patient"
        )["Weeks"].transform("min")
        self.df_train["FVC_Baseline"] = self.df_train.groupby("Patient")[
            "FVC"
        ].transform("first")

        for patient in self.df_test["Patient"].unique():
            self.df_submission.loc[
                self.df_submission["Patient"] == patient, "FVC_Baseline"
            ] = self.df_test.loc[self.df_test["Patient"] == patient, "FVC"].values[0]
            self.df_submission.loc[
                self.df_submission["Patient"] == patient, "Percent"
            ] = self.df_test.loc[self.df_test["Patient"] == patient, "Percent"].values[
                0
            ]
            self.df_submission.loc[self.df_submission["Patient"] == patient, "Age"] = (
                self.df_test.loc[self.df_test["Patient"] == patient, "Age"].values[0]
            )
            self.df_submission.loc[self.df_submission["Patient"] == patient, "Sex"] = (
                self.df_test.loc[self.df_test["Patient"] == patient, "Sex"].values[0]
            )
            self.df_submission.loc[
                self.df_submission["Patient"] == patient, "SmokingStatus"
            ] = self.df_test.loc[
                self.df_test["Patient"] == patient, "SmokingStatus"
            ].values[
                0
            ]

        base_week_map = self.df_test.set_index("Patient")["Weeks"].to_dict()
        self.df_submission["Weeks_Passed"] = self.df_submission[
            "Weeks"
        ] - self.df_submission["Patient"].map(base_week_map).astype(np.int16)

        self.df_all = pd.concat(
            [self.df_train, self.df_submission], ignore_index=True, axis=0
        )
        self.df_all["Age"] = self.df_all["Age"] + np.int8(
            np.floor(self.df_all["Weeks_Passed"] / 52)
        )

        self.df_all["Weeks"] = self.df_all["Weeks"].astype(np.int16)
        self.df_all["Age"] = self.df_all["Age"].astype(np.float32)
        self.df_all["FVC_Baseline"] = self.df_all["FVC_Baseline"].astype(np.float32)
        self.df_all["Percent"] = self.df_all["Percent"].astype(np.float32)
        self.df_all["Weeks_Passed"] = self.df_all["Weeks_Passed"].astype(np.float32)
        self.df_all["Sex"] = self.df_all["Sex"].astype(np.uint8)
        self.df_all["SmokingStatus"] = self.df_all["SmokingStatus"].astype(np.uint8)
        self.df_all["FVC"] = self.df_all["FVC"].astype(np.float32)

        self.df_train = self.df_all.loc[self.df_all["Type"] == "Train", :].drop(
            columns=["Type"]
        )
        self.df_test = self.df_all.loc[self.df_all["Type"] == "Test", :].drop(
            columns=["Type", "FVC", "CV1_Fold", "CV2_Fold", "CV3_Fold"]
        )

    def get_data(self):
        self._drop_duplicates()
        self._label_encode()
        self._create_folds()
        self._create_baseline_features()

        print(f"Preprocessed Training Set Shape = {self.df_train.shape}")
        print(f"Preprocessed Test Set Shape = {self.df_test.shape}")
        return self.df_train.copy(deep=True), self.df_test.copy(deep=True)




In [4]:
preprocessor_parameters = {
    "df_train": df_train,
    "df_test": df_test,
    "df_submission": df_submission,
    "n_folds": 2,
    "shuffle": True,
    "resize_shape": (512, 512),
}

preprocessor = Preprocessor(**preprocessor_parameters)
df_train, df_test = preprocessor.get_data()




Preprocessed Training Set Shape = (1387, 12)
Preprocessed Test Set Shape = (1908, 8)


In [5]:
class QuantileRegressorMLP:
    """
    Dependency-free fallback that preserves the same public API and prediction-column semantics.
    """

    def __init__(self, model, predictors, mlp_parameters, qr_parameters):
        self.model = model
        self.predictors = predictors
        self.mlp_parameters = mlp_parameters
        self.qr_parameters = qr_parameters

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        y_true = np.asarray(y_true).reshape(-1)
        y_pred = np.asarray(y_pred).reshape(-1)
        sigma = np.asarray(sigma).reshape(-1)

        sigma_clipped = np.maximum(sigma, 70)
        delta_clipped = np.minimum(np.abs(y_true - y_pred), 1000)
        score = -np.sqrt(2) * delta_clipped / sigma_clipped - np.log(
            np.sqrt(2) * sigma_clipped
        )
        return np.mean(score)

    def _fit_patient_models(self, tr_df):
        patient_models = {}
        patient_sigma = {}
        residuals_all = []

        for p, g in tr_df.groupby("Patient"):
            g = g.sort_values("Weeks_Passed")
            x = g["Weeks_Passed"].values.reshape(-1, 1).astype(np.float64)
            y = g["FVC"].values.astype(np.float64)

            if len(g) >= 2 and np.std(x) > 0:
                reg = LinearRegression().fit(x, y)
                a = float(reg.intercept_)
                b = float(reg.coef_[0])
                y_hat = reg.predict(x)
                patient_models[p] = ("lin", a, b)
            else:
                mu = float(np.mean(y))
                y_hat = np.full_like(y, mu, dtype=np.float64)
                patient_models[p] = ("const", mu, 0.0)

            resid = y - y_hat
            residuals_all.append(resid)

            if len(resid) >= 3:
                s = float(np.std(resid))
            else:
                s = np.nan
            patient_sigma[p] = s

        residuals_all = (
            np.concatenate(residuals_all) if len(residuals_all) else np.array([0.0])
        )
        global_sigma = float(np.std(residuals_all))
        if (not np.isfinite(global_sigma)) or global_sigma < 70:
            global_sigma = 200.0

        for p in list(patient_sigma.keys()):
            s = patient_sigma[p]
            if (not np.isfinite(s)) or s < 1e-6:
                patient_sigma[p] = global_sigma

        return patient_models, patient_sigma, global_sigma

    def _predict_with_models(
        self,
        df,
        patient_models,
        patient_sigma,
        global_sigma,
        sigma_scale,
        slope_shrink,
        fvc_bias,
    ):
        patients = df["Patient"].to_numpy()
        weeks_passed = df["Weeks_Passed"].to_numpy(dtype=np.float64)

        a = np.empty(len(df), dtype=np.float64)
        b = np.empty(len(df), dtype=np.float64)
        s0 = np.empty(len(df), dtype=np.float64)

        for i, p in enumerate(patients):
            if p in patient_models:
                _, ai, bi = patient_models[p]
                a[i] = ai
                b[i] = bi
                s0[i] = float(patient_sigma.get(p, global_sigma))
            else:
                base = float(df.iloc[i].get("FVC_Baseline", np.nan))
                if not np.isfinite(base):
                    base = float(df_train["FVC"].median())
                a[i] = base
                b[i] = 0.0
                s0[i] = float(global_sigma)

        pred = a + (b * float(slope_shrink)) * weeks_passed + float(fvc_bias)

        sigma = s0 * float(sigma_scale)
        sigma = np.where(np.isfinite(sigma), sigma, 200.0)
        sigma = np.maximum(sigma, 70.0)

        return pred, sigma

    def train(self, X_train, y_train):
        tr = df_train[["Patient", "Weeks_Passed", "FVC", "CV3_Fold"]].copy()

        self.patient_models, self.patient_sigma, self.global_sigma = (
            self._fit_patient_models(tr)
        )

        y_true = tr["FVC"].to_numpy(dtype=np.float64)
        oof_pred = np.empty(len(tr), dtype=np.float64)
        oof_sigma_base = np.empty(len(tr), dtype=np.float64)

        # CHANGE (score): store fold-aligned (a,b) used for each OOF row, so grid-search
        # tunes the same fold-specific models that produced oof_pred/oof_sigma_base.
        oof_a = np.empty(len(tr), dtype=np.float64)
        oof_b = np.empty(len(tr), dtype=np.float64)

        folds = tr["CV3_Fold"].to_numpy(dtype=np.uint8)

        for fold in np.unique(folds):
            train_mask = folds != fold
            valid_mask = folds == fold

            tr_fit = tr.loc[train_mask, ["Patient", "Weeks_Passed", "FVC"]].copy()
            pm, ps, gs = self._fit_patient_models(tr_fit)

            df_valid = tr.loc[valid_mask, ["Patient", "Weeks_Passed"]].copy()
            df_valid["FVC_Baseline"] = np.nan
            pred_v, sigma_v = self._predict_with_models(
                df_valid,
                pm,
                ps,
                gs,
                sigma_scale=1.0,
                slope_shrink=1.0,
                fvc_bias=0.0,
            )

            idxs = np.where(valid_mask)[0]
            oof_pred[idxs] = pred_v
            oof_sigma_base[idxs] = sigma_v

            # fold-aligned intercept/slope per row
            patients_v = df_valid["Patient"].to_numpy()
            a_v = np.empty(len(df_valid), dtype=np.float64)
            b_v = np.empty(len(df_valid), dtype=np.float64)
            for i, p in enumerate(patients_v):
                if p in pm:
                    _, ai, bi = pm[p]
                    a_v[i] = float(ai)
                    b_v[i] = float(bi)
                else:
                    a_v[i] = float(np.nanmean(y_true))
                    b_v[i] = 0.0
            oof_a[idxs] = a_v
            oof_b[idxs] = b_v

        candidate_scales = np.array(
            [0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.25, 1.35, 1.50, 1.70, 1.90],
            dtype=np.float64,
        )
        candidate_shrinks = np.array(
            [0.90, 0.93, 0.95, 0.96, 0.97, 0.98, 1.00],
            dtype=np.float64,
        )
        candidate_bias = np.array(
            [-120.0, -80.0, -50.0, -25.0, 0.0, 25.0, 50.0, 80.0, 120.0],
            dtype=np.float64,
        )

        best_score = -1e18
        best_shrink = 0.95
        best_scale = 1.0
        best_bias = 0.0

        weeks_oof = tr["Weeks_Passed"].to_numpy(dtype=np.float64)

        # CHANGE (score): use fold-aligned (oof_a,oof_b) rather than full-data patient_models.
        for shrink in candidate_shrinks:
            pred_s = oof_a + (oof_b * float(shrink)) * weeks_oof
            for bias in candidate_bias:
                pred_sb = pred_s + float(bias)
                for scale in candidate_scales:
                    score = self.laplace_log_likelihood_metric(
                        y_true,
                        pred_sb,
                        oof_sigma_base * float(scale),
                    )
                    if score > best_score:
                        best_score = float(score)
                        best_shrink = float(shrink)
                        best_scale = float(scale)
                        best_bias = float(bias)

        self.slope_shrink = best_shrink
        self.sigma_scale = best_scale
        self.fvc_bias = best_bias

        for cv in range(1, 4):
            fvc_col = f"CV{cv}_MLP_FVC_Predictions"
            conf_col = f"CV{cv}_MLP_Confidence_Predictions"
            df_train[fvc_col] = np.nan
            df_train[conf_col] = np.nan
            for q in self.qr_parameters["quantiles"]:
                df_train[f"CV{cv}_QR_{q}_Predictions"] = np.nan

        df_tr_pred = df_train[["Patient", "Weeks_Passed", "FVC_Baseline"]].copy()
        y_pred_full, sigma_full = self._predict_with_models(
            df_tr_pred,
            self.patient_models,
            self.patient_sigma,
            self.global_sigma,
            sigma_scale=self.sigma_scale,
            slope_shrink=self.slope_shrink,
            fvc_bias=self.fvc_bias,
        )

        q_width = 0.67448975 * sigma_full

        for cv in range(1, 4):
            df_train[f"CV{cv}_MLP_FVC_Predictions"] = y_pred_full.astype(np.float32)
            df_train[f"CV{cv}_MLP_Confidence_Predictions"] = sigma_full.astype(
                np.float32
            )
            df_train[f"CV{cv}_QR_0.5_Predictions"] = y_pred_full.astype(np.float32)
            df_train[f"CV{cv}_QR_0.25_Predictions"] = (y_pred_full - q_width).astype(
                np.float32
            )
            df_train[f"CV{cv}_QR_0.75_Predictions"] = (y_pred_full + q_width).astype(
                np.float32
            )

        approx_oof_metric = self.laplace_log_likelihood_metric(
            y_true,
            (oof_a + (oof_b * self.slope_shrink) * weeks_oof) + self.fvc_bias,
            np.maximum(oof_sigma_base * self.sigma_scale, 70.0),
        )
        approx_train_metric = self.laplace_log_likelihood_metric(
            df_train["FVC"].to_numpy(dtype=np.float64),
            df_train["CV1_MLP_FVC_Predictions"].to_numpy(dtype=np.float64),
            df_train["CV1_MLP_Confidence_Predictions"].to_numpy(dtype=np.float64),
        )
        print(
            f"Fallback patient-linear training done. "
            f"OOF-selected sigma_scale={self.sigma_scale:.3f}, slope_shrink={self.slope_shrink:.2f}, fvc_bias={self.fvc_bias:.1f}. "
            f"Approx OOF metric: {approx_oof_metric:.6f} | Approx train metric: {approx_train_metric:.6f}"
        )

    def predict(self, X_test):
        for cv in range(1, 4):
            X_test[f"CV{cv}_MLP_FVC_Predictions"] = np.nan
            X_test[f"CV{cv}_MLP_Confidence_Predictions"] = np.nan
            for q in self.qr_parameters["quantiles"]:
                X_test[f"CV{cv}_QR_{q}_Predictions"] = np.nan

        df_te_pred = X_test[["Patient", "Weeks_Passed", "FVC_Baseline"]].copy()
        pred, sigma = self._predict_with_models(
            df_te_pred,
            self.patient_models,
            self.patient_sigma,
            self.global_sigma,
            sigma_scale=self.sigma_scale,
            slope_shrink=self.slope_shrink,
            fvc_bias=self.fvc_bias,
        )

        q_width = 0.67448975 * sigma
        q25 = pred - q_width
        q75 = pred + q_width

        for cv in range(1, 4):
            X_test[f"CV{cv}_MLP_FVC_Predictions"] = pred.astype(np.float32)
            X_test[f"CV{cv}_MLP_Confidence_Predictions"] = sigma.astype(np.float32)
            X_test[f"CV{cv}_QR_0.5_Predictions"] = pred.astype(np.float32)
            X_test[f"CV{cv}_QR_0.25_Predictions"] = q25.astype(np.float32)
            X_test[f"CV{cv}_QR_0.75_Predictions"] = q75.astype(np.float32)




In [6]:
X_train = df_train.drop(columns=["FVC", "Weeks"])  # keep CV*_Fold columns
y_train = df_train["FVC"].copy(deep=True)

model_parameters = {
    "model": "Stack",
    "predictors": [
        "Age",
        "Sex",
        "SmokingStatus",
        "FVC_Baseline",
        "Percent",
        "Weeks_Passed",
    ],
    "mlp_parameters": {"lr": 0.0005, "epochs": 350, "batch_size": 2**5},
    "qr_parameters": {
        "quantiles": [0.25, 0.5, 0.75],
        "lr": 0.0005,
        "epochs": 150,
        "batch_size": 2**5,
    },
}

qr_mlp = QuantileRegressorMLP(**model_parameters)
qr_mlp.train(X_train, y_train)
qr_mlp.predict(df_test)

print(
    "df_test prediction columns example:",
    [c for c in df_test.columns if "Predictions" in c][:10],
)




Fallback patient-linear training done. OOF-selected sigma_scale=1.900, slope_shrink=0.90, fvc_bias=-120.0. Approx OOF metric: -9.198459 | Approx train metric: -6.638111
df_test prediction columns example: ['CV1_MLP_FVC_Predictions', 'CV1_MLP_Confidence_Predictions', 'CV1_QR_0.25_Predictions', 'CV1_QR_0.5_Predictions', 'CV1_QR_0.75_Predictions', 'CV2_MLP_FVC_Predictions', 'CV2_MLP_Confidence_Predictions', 'CV2_QR_0.25_Predictions', 'CV2_QR_0.5_Predictions', 'CV2_QR_0.75_Predictions']


In [7]:
class SubmissionPipeline:
    def __init__(self, df_train, df_test, sample_submission):
        self.df_train = df_train.copy(deep=True)
        self.df_test = df_test.copy(deep=True)
        self.sample_submission = sample_submission.copy(deep=True)

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        sigma_clipped = np.maximum(sigma, 70)
        delta_clipped = np.minimum(np.abs(y_true - y_pred), 1000)
        score = -np.sqrt(2) * delta_clipped / sigma_clipped - np.log(
            np.sqrt(2) * sigma_clipped
        )
        return np.mean(score)

    def single_model(self, model_key):
        parts = model_key.split("_")
        if len(parts) != 2:
            raise ValueError("model_key must look like 'CV1_MLP' or 'CV1_QR'")

        cv, mtype = parts[0], parts[1]
        self.df_test["Patient_Week"] = (
            self.df_test["Patient"].astype(str)
            + "_"
            + self.df_test["Weeks"].astype(str)
        )

        if mtype == "MLP":
            fvc_col = f"{cv}_MLP_FVC_Predictions"
            conf_col = f"{cv}_MLP_Confidence_Predictions"
            score = self.laplace_log_likelihood_metric(
                self.df_train["FVC"].values,
                self.df_train[fvc_col].values,
                self.df_train[conf_col].values,
            )
            print(f"Single Model {model_key} Score: {score:.6f}")

            pred = pd.DataFrame(
                {
                    "Patient_Week": self.df_test["Patient_Week"].values,
                    "FVC": self.df_test[fvc_col].values,
                    "Confidence": self.df_test[conf_col].values,
                }
            )

        elif mtype == "QR":
            q = [0.25, 0.5, 0.75]
            lo_col = f"{cv}_QR_{q[0]}_Predictions"
            mid_col = f"{cv}_QR_{q[1]}_Predictions"
            hi_col = f"{cv}_QR_{q[2]}_Predictions"

            iqr_width_train = (
                self.df_train[hi_col].values - self.df_train[lo_col].values
            )
            sigma_train = iqr_width_train / (2.0 * 0.67448975)

            score = self.laplace_log_likelihood_metric(
                self.df_train["FVC"].values,
                self.df_train[mid_col].values,
                sigma_train,
            )
            print(f"Single Model {model_key} Score: {score:.6f}")

            iqr_width_test = self.df_test[hi_col].values - self.df_test[lo_col].values
            sigma_test = iqr_width_test / (2.0 * 0.67448975)

            pred = pd.DataFrame(
                {
                    "Patient_Week": self.df_test["Patient_Week"].values,
                    "FVC": self.df_test[mid_col].values,
                    "Confidence": sigma_test,
                }
            )
        else:
            raise ValueError(f"Unknown model type: {mtype}")

        out = self.sample_submission[["Patient_Week"]].merge(
            pred, on="Patient_Week", how="left"
        )

        if out["FVC"].isna().any() or out["Confidence"].isna().any():
            fallback = self.df_test[["Patient", "Weeks", "FVC_Baseline"]].copy()
            fallback["Patient_Week"] = (
                fallback["Patient"].astype(str) + "_" + fallback["Weeks"].astype(str)
            )
            out = out.drop(columns=["FVC", "Confidence"]).merge(
                fallback[["Patient_Week", "FVC_Baseline"]],
                on="Patient_Week",
                how="left",
            )
            out["FVC"] = out["FVC_Baseline"].fillna(out["FVC_Baseline"].median())
            out["Confidence"] = 200.0
            out = out[["Patient_Week", "FVC", "Confidence"]]

        out["Confidence"] = (
            pd.to_numeric(out["Confidence"], errors="coerce")
            .fillna(200.0)
            .astype(np.float32)
            .clip(lower=70)
        )
        out["FVC"] = pd.to_numeric(out["FVC"], errors="coerce").astype(np.float32)

        print(out.describe(include="all"))
        return out




In [8]:
sub = SubmissionPipeline(df_train, df_test, df_submission)
final_submission = sub.single_model("CV3_MLP")
final_submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", final_submission.shape)
print(final_submission.head())

Single Model CV3_MLP Score: -6.638112
                         Patient_Week          FVC   Confidence
count                            1908  1908.000000  1908.000000
unique                           1908          NaN          NaN
top     ID00047637202184938901501_102          NaN          NaN
freq                                1          NaN          NaN
mean                              NaN  2924.055664   229.122253
std                               NaN  1073.814819     0.002320
min                               NaN  1463.000000   229.122208
25%                               NaN  2207.000000   229.122208
50%                               NaN  2987.000000   229.122208
75%                               NaN  3454.000000   229.122208
max                               NaN  6279.000000   229.122208
Wrote submission.csv with shape: (1908, 3)
                   Patient_Week     FVC  Confidence
0  ID00126637202218610655908_-3  2255.0  229.122208
1  ID00126637202218610655908_-2  2255.0  229.12

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
